In [4]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel
from torch.nn.functional import softmax
from datasets import load_dataset


/Users/plato/code/interpret/v/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [6]:
LABELS = ['negative', 'positive']
TOKENIZED_LABELS = list(tokenizer(LABELS, return_tensors='pt', padding=True, truncation=True)['input_ids'].squeeze().tolist())
LABEL_MAPPING = {'negative': 0, 'positive': 1}


In [7]:
imdb = load_dataset('imdb')
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
imdb_test = imdb['test'].shuffle().select(range(1000))
# imdb_test['label']

In [97]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2Tokenizer, GPT2Model
from datasets import load_dataset
from tqdm import tqdm

class SimpleGPT2SequenceClassifier(nn.Module):
    def __init__(self, hidden_size: int, num_classes: int, max_seq_len: int, gpt_model_name: str):
        super(SimpleGPT2SequenceClassifier, self).__init__()
        self.gpt2model = GPT2Model.from_pretrained(gpt_model_name)
        self.fc1 = nn.Linear(hidden_size * max_seq_len, num_classes)

    def forward(self, input_id, mask):
        gpt_out = self.gpt2model(input_ids=input_id, attention_mask=mask).last_hidden_state
        batch_size = gpt_out.shape[0]
        linear_output = self.fc1(gpt_out.view(batch_size, -1))
        return linear_output

# Load the IMDB dataset
dataset = load_dataset('imdb')

# Load the GPT-2 tokenizer
max_seq_len = 1024

# Preprocess the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=max_seq_len)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Convert dataset to PyTorch tensors
class IMDBDataset(Dataset):
    def __init__(self, tokenized_dataset):
        self.tokenized_dataset = tokenized_dataset

    def __len__(self):
        return len(self.tokenized_dataset)

    def __getitem__(self, idx):
        item = self.tokenized_dataset[idx]
        input_ids = torch.tensor(item['input_ids'])
        attention_mask = torch.tensor(item['attention_mask'])
        label = torch.tensor(item['label'])
        return input_ids, attention_mask, label

train_dataset = IMDBDataset(tokenized_datasets['train'])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

hidden_size = 768  # GPT-2 hidden size
num_classes = 2
model = SimpleGPT2SequenceClassifier(hidden_size=hidden_size, num_classes=num_classes, max_seq_len=max_seq_len, gpt_model_name='gpt2')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

# Training loop
num_epochs = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for input_ids, attention_mask, labels in tqdm(train_loader):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss}")

# Evaluation loop
model.eval()
total_correct = 0
total_samples = 0

  0%|          | 0/3125 [00:31<?, ?it/s]


KeyboardInterrupt: 

In [9]:
def classify_sentiment(model, batch):
    with torch.no_grad():
        outputs = model(input_ids=batch)
    logits = outputs.logits[:, -1, :]
    probabilities = softmax(logits, dim=-1)
    probabilities = probabilities[:, TOKENIZED_LABELS]
    predicted_idxs = torch.argmax(probabilities, dim=-1)
    predicted_sentiments = [LABELS[idx.item()] for idx in predicted_idxs]
    return predicted_sentiments

def classify_sentiment_generate(model, batch):
    outputs = model.generate(input_ids=batch, max_length=batch.shape[1] + 1, num_return_sequences=1)
    decoded_outputs = [tokenizer.decode(output) for output in outputs]
    predicted_sentiments = []
    for decoded_output in decoded_outputs:
        last_word = decoded_output.split()[-1].lower()
        if last_word in LABELS:
            predicted_sentiments.append(last_word)
        else:
            predicted_sentiments.append(classify_sentiment_random(model, [decoded_output])[0])
    return predicted_sentiments

def classify_sentiment_random(model, batch):
    return ['positive' if torch.rand(1) < 0.5 else 'negative' for _ in batch]

In [15]:
from torch.utils.data import DataLoader

def test_classification_methods(model, dataset, methods):
    results = {method_name: {'correct': 0, 'total': 0} for method_name in methods.keys()}

    for batch_samples in dataset:
        batch_texts = [sample + '\n\n the sentiment of this review is ' for sample in batch_samples['text']]
        batch_labels = ['positive' if sample == 1 else 'negative' for sample in batch_samples['label']]

        batch_tokens = tokenizer(batch_texts, return_tensors='pt', max_length=1024, truncation=True, padding=True)['input_ids']

        for method_name, method in methods.items():
            predicted_labels = method(model, batch_tokens)
            for predicted_label, true_label in zip(predicted_labels, batch_labels):
                results[method_name]['correct'] += 1 if predicted_label == true_label else 0
                results[method_name]['total'] += 1

    for method_name in results.keys():
        accuracy = results[method_name]['correct'] / results[method_name]['total']
        print(f"{method_name} accuracy: {accuracy:.2%}")

    return results


imdb_test_dl = DataLoader(imdb_test, batch_size=16, shuffle=False)
methods = {
    'logits': classify_sentiment,
    'random': classify_sentiment_random
}

test_classification_methods(model, imdb_test_dl, methods)


logits accuracy: 48.80%
random accuracy: 48.10%
